In [3]:
from cassandra.cluster import Cluster, Session
from cassandra.auth import PlainTextAuthProvider
import json

DependencyException: Unable to load a default connection class
The following exceptions were observed: 
 - The C extension needed to use libev was not found.  This probably means that you didn't have the required build dependencies when installing the driver.  See http://datastax.github.io/python-driver/installation.html#c-extensions for instructions on installing build dependencies and building the C extension.
 - Unable to import asyncore module.  Note that this module has been removed in Python 3.12 so when using the driver with this version (or anything newer) you will need to use one of the other event loop implementations.

In [2]:
cloud_config = {
    'secure_connect_bundle': '../cassandra/secure-connect-test-cassandra.zip'
}

with open("../cassandra/cassandra-token.json") as f:
    secrets = json.load(f)

CLIENT_ID = secrets["clientId"]
CLIENT_SECRET = secrets["secret"]
TOKEN = secrets["token"]

KEYSPACE = "proyecto_recomendaciones"

In [ ]:
auth_provider = PlainTextAuthProvider(CLIENT_ID, CLIENT_SECRET)
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
session: Session = cluster.connect()

# Verificar conexión
row = session.execute("SELECT release_version FROM system.local").one()
if row:
    print(f"Cassandra Release Version: {row[0]}")
else:
    print("An error occurred.")

# Establecer el keyspace
session.set_keyspace(KEYSPACE)

Cassandra Release Version: 4.0.11-3a52ecb0d31d


# IMPORTAR DATOS A CASSANDRA

In [2]:
!pip install cassandra-driver

  Created wheel for cassandra-driver: filename=cassandra_driver-3.29.2-cp37-cp37m-win_amd64.whl size=333097 sha256=3e177927d033ef1ac2b96261893cfb58d48cefc087199f4dbd132b4a40027cc5
  Stored in directory: c:\users\victo\appdata\local\pip\cache\wheels\9e\85\4e\c44bc764240ddd975b9101218b11d7e7a87e51c8603e18c21b
Successfully built cassandra-driver


In [4]:
import csv
import json
import time
from cassandra.cluster import Cluster, Session
from cassandra.auth import PlainTextAuthProvider
from cassandra.concurrent import execute_concurrent_with_args
from concurrent.futures import ThreadPoolExecutor

# === CONFIGURACIÓN DE CONEXIÓN ===
cloud_config = {
    'secure_connect_bundle': '../cassandra/secure-connect-test-cassandra.zip'
}

with open("../cassandra/cassandra-token.json") as f:
    secrets = json.load(f)

auth_provider = PlainTextAuthProvider(secrets["clientId"], secrets["secret"])
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
session: Session = cluster.connect()
session.set_keyspace("proyecto_recomendaciones")

session.execute("""
    CREATE TABLE IF NOT EXISTS recomendaciones_ALS (
        userId int,
        movieId int,
        prediction float,
        title text,
        PRIMARY KEY (userId, movieId)
    )
""")

# === PREPARE STATEMENT ===
insert_stmt = session.prepare("""
    INSERT INTO recomendaciones_ALS (userId, movieId, prediction, title)
    VALUES (?, ?, ?, ?)
""")

# === PARÁMETROS DE PROCESAMIENTO ===
CSV_PATH = "../Predicciones_ALS.csv"
CHUNK_SIZE = 1000     # Cantidad de filas por lote
MAX_WORKERS = 8       # Hilos paralelos

# === FUNCIÓN DE INSERCIÓN CONCURRENTE ===
def insertar_datos(session: Session, insert_stmt, data_chunk):
    execute_concurrent_with_args(
        session,
        insert_stmt,
        data_chunk,
        concurrency=MAX_WORKERS
    )

# === PROCESAR CSV EN BLOQUES ===
def procesar_csv_grande():
    total = 0
    start_time = time.time()
    with open(CSV_PATH, newline='', encoding="utf-8") as csvfile:
        reader = csv.DictReader(csvfile)
        chunk = []

        for row in reader:
            try:
                data = (
                    int(row["userId"]),
                    int(row["movieId"]),
                    float(row["prediction"]),
                    row["title"]
                )
                chunk.append(data)
            except Exception as e:
                print(f"Error parseando fila: {e}")
                continue

            if len(chunk) >= CHUNK_SIZE:
                insertar_datos(session, insert_stmt, chunk)
                total += len(chunk)
                print(f"{total} filas insertadas...")
                chunk.clear()

        # Insertar las últimas filas
        if chunk:
            insertar_datos(session, insert_stmt, chunk)
            total += len(chunk)
            print(f"{total} filas insertadas (final).")

    print(f"✅ Carga completa en {time.time() - start_time:.2f} segundos.")

# === EJECUTAR ===
procesar_csv_grande()


1000 filas insertadas...
2000 filas insertadas...
3000 filas insertadas...
4000 filas insertadas...
5000 filas insertadas...
6000 filas insertadas...
7000 filas insertadas...
8000 filas insertadas...
9000 filas insertadas...
10000 filas insertadas...
11000 filas insertadas...
12000 filas insertadas...
13000 filas insertadas...
14000 filas insertadas...
15000 filas insertadas...
16000 filas insertadas...
17000 filas insertadas...
18000 filas insertadas...
19000 filas insertadas...
20000 filas insertadas...
21000 filas insertadas...
22000 filas insertadas...
23000 filas insertadas...
24000 filas insertadas...
25000 filas insertadas...
26000 filas insertadas...
27000 filas insertadas...
28000 filas insertadas...
29000 filas insertadas...
30000 filas insertadas...
31000 filas insertadas...
32000 filas insertadas...
33000 filas insertadas...
34000 filas insertadas...
35000 filas insertadas...
36000 filas insertadas...
37000 filas insertadas...
38000 filas insertadas...
39000 filas insertada

KeyboardInterrupt: 

# LEER DATOS DE CASSANDRA